In [1]:
# Check the right kernel is being used 

import sys
sys.executable

'/workspaces/F1-Qualifying-Dashboard/.venv/bin/python'

In [48]:
import duckdb

con = duckdb.connect("../data/processed/f1.duckdb")

con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_name = 'qualifying_laps'
""").fetchall()

[('main', 'qualifying_laps')]

In [2]:
# Packages 

import duckdb

# Establish a connection to the DuckDB file that dbt writes models into.
# The path is relative to the directory (data/processed/)
con = duckdb.connect("../data/processed/f1.duckdb")

In [10]:
# Run a SQL query against the database.
# Check a few rows of data

df = con.execute("select * from qualifying_laps limit 20").df()

df.head()

,driver,session,lap_time,speed,year,round,track
0,HAM,Q,82.681,311.0,2019,1,Australian Grand Prix
1,HAM,Q,82.043,316.0,2019,1,Australian Grand Prix
2,HAM,Q,81.861,315.0,2019,1,Australian Grand Prix
3,HAM,Q,81.014,316.0,2019,1,Australian Grand Prix
4,HAM,Q,81.055,319.0,2019,1,Australian Grand Prix


In [ ]:
# Run a SQL query against the database.
# Making a check in here to compare the results I got from first sample with official reports
# VER fastest time in Sao Paulo was 1:10.162 that is 70.162 seconds 
# https://www.formula1.com/en/results/2023/races/1224/brazil/qualifying

df = con.execute("select * from qualifying_laps where driver = 'VER' \
                    and lap_time = (select min(lap_time) from qualifying_laps where driver = 'VER') \
                  limit 20").df()

df.head()

,driver,session,lap_time,lap_delta,speed,year,round,track
0,VER,Q,70.162,0.141,330.0,2023,20,São Paulo Grand Prix


In [19]:
# Run a SQL query against the database.
# Making a check in here to compare the results I got from first sample with official reports

df = con.execute("select min(lap_time) from qualifying_laps where driver = 'LEC' \
                    and track = 'Bahrain Grand Prix' and year = 2022 \
                  limit 20").df()

df.head()

,min(lap_time)
0,90.558


In [3]:
# Run a SQL query against the database.
# Check a few rows of data

df = con.execute("select count(*) from qualifying_laps").df()

df.head()

,count_star()
0,10119


In [8]:
# Run a SQL query against the database.
# Check a few rows of data

df = con.execute("select year, count(round) as n_rounds from qualifying_laps \
                    group by 1").df()

df.head(10)

,year,n_rounds
0,2019,1949
1,2020,1675
2,2021,2125
3,2022,1964
4,2023,2089
5,2026,317


In [ ]:
# Run a SQL query against the database.
# Check for null values

df = con.execute("select year, count(*) as n_nulls from qualifying_laps \
                    where lap_delta is null group by 1").df()

df.head()

,year,n_nulls


In [4]:
# Run a SQL query against the database.
# Check a few rows of data for stg_qualifying_laps

df = con.execute("select * from main_staging.stg_qualifying_laps").df()

df.head()

,driver,session,lap_time,lap_delta,speed,year,round,track
0,HAM,Q,82.681,2.195,311.0,2019,1,Australian Grand Prix
1,HAM,Q,82.043,1.557,316.0,2019,1,Australian Grand Prix
2,HAM,Q,81.861,1.375,315.0,2019,1,Australian Grand Prix
3,HAM,Q,81.014,0.528,316.0,2019,1,Australian Grand Prix
4,HAM,Q,81.055,0.569,319.0,2019,1,Australian Grand Prix


In [6]:
# Run a SQL query against the database.
# Get a list of drivers and year

df = con.execute("select distinct driver, year from main_staging.stg_qualifying_laps").df()

# Write to CSV
df.to_csv("drivers_years.csv", index=False)

In [7]:
# Run a SQL query against the database.
# Check a few rows of data for int_qualifying_laps_with_teams

df = con.execute("select * from main.int_qualifying_laps_with_teams").df()

df.head()

,driver,session,lap_time,lap_delta,speed,year,round,track,team
0,HAM,Q,82.681,2.195,311.0,2019,1,Australian Grand Prix,Mercedes
1,HAM,Q,82.043,1.557,316.0,2019,1,Australian Grand Prix,Mercedes
2,HAM,Q,81.861,1.375,315.0,2019,1,Australian Grand Prix,Mercedes
3,HAM,Q,81.014,0.528,316.0,2019,1,Australian Grand Prix,Mercedes
4,HAM,Q,81.055,0.569,319.0,2019,1,Australian Grand Prix,Mercedes


In [49]:
# Run a SQL query against the database.
# Check a few rows of data for best_laps_summary

df = con.execute("select * from main.best_laps_summary where year = 2022 and track = 'Bahrain Grand Prix'").df()

df.head()

,driver,team,year,track,fastest_lap_time,fastest_lap_time_formatted,speed_on_fastest_lap,session,lap_delta
0,LAT,Williams,2022,Bahrain Grand Prix,93.634,1.:33:634,319.0,Q,3.076
1,STR,Aston Martin,2022,Bahrain Grand Prix,93.032,1.:33:32.,311.0,Q,2.474
2,RIC,McLaren,2022,Bahrain Grand Prix,92.945,1.:32:945,311.0,Q,2.387
3,HUL,Aston Martin,2022,Bahrain Grand Prix,92.777,1.:32:777,310.0,Q,2.219
4,TSU,AlphaTauri,2022,Bahrain Grand Prix,92.750,1.:32:750,318.0,Q,2.192


In [23]:
# Run a SQL query against the database.
# Check a few rows of data for best_laps_summary

df = con.execute("select count(*) from main.best_laps_summary").df()

df.head()

,count_star()
0,2056


In [50]:
# Run a SQL query against the database.
# Get the final dataset from my mart model created with dbt

df = con.execute("select * from main.best_laps_summary").df()

# Write to CSV
df.to_csv("best_lap_summary.csv", index=False, encoding='utf-8-sig')

In [47]:
# Close the connection 
con.close()